# Block 3 — Nonverbal marks & verbal drafts

Reads a Block 1 ZIP. Emits `hypotheses.json`. **Does not** run KG `assume()` or invent tube counts (`prior_expected` is Block 4, and even there empty crops stay empty).

Uses `backend='lexicon'` so Colab does not need Paddle/TrOCR. No PHI.


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V4 — zipball (no git, no %pip -e; editable install restarts Colab mid-cell)
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

if not (SRC / "med_doc" / "__init__.py").is_file():
    zpath = CONTENT / "epq3-block1.zip"
    print("Downloading", URL)
    urllib.request.urlretrieve(URL, zpath)
    extract = CONTENT / "_epq3_extract"
    if extract.exists():
        shutil.rmtree(extract)
    extract.mkdir()
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(extract)
    found = list(extract.glob("*/src/med_doc/__init__.py"))
    if not found:
        raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
    unpacked = found[0].parents[2]
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.move(str(unpacked), str(REPO))
    shutil.rmtree(extract, ignore_errors=True)
    zpath.unlink(missing_ok=True)

sys.path.insert(0, str(SRC.resolve()))
os.chdir(REPO)
import med_doc

print("BOOTSTRAP_V4")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.htr.batch import process_from_block1
from med_doc.kg import KnowledgeGraph
from med_doc.normalization.batch import normalize_batch
from med_doc.rescoring import process_from_block3
from med_doc.review import ReviewPatch, process_from_block4

kg = KnowledgeGraph.load()

def ensure_block1():
    z = OUT / "block1.zip"
    if z.exists():
        return z
    return Path(normalize_batch([demo_sheet()], output_dir=OUT / "b1", output_zip=z)["output_zip"])

def ensure_block3():
    z = OUT / "block3.zip"
    if z.exists():
        return z
    b1 = ensure_block1()
    return Path(process_from_block1(b1, output_dir=OUT / "b3", output_zip=z, kg=kg, backend="lexicon", mode="both")["output_zip"])

def ensure_block4():
    z = OUT / "block4.zip"
    if z.exists():
        return z
    b3 = ensure_block3()
    return Path(process_from_block3(b3, output_dir=OUT / "b4", output_zip=z, kg=kg)["output_zip"])


## 1. Block 1 ZIP → hypotheses


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

b3 = process_from_block1(
    ensure_block1(),
    output_dir=OUT / "b3",
    output_zip=OUT / "block3.zip",
    kg=kg,
    backend="lexicon",
    mode="both",
)
print("docs", b3["manifest"]["total_documents"], "hitl", b3["manifest"]["hitl_documents"])
doc = b3["manifest"]["documents"][0]
print("doc_id", doc["doc_id"], "ticked", doc["ticked_test_ids"][:20])
hyp_path = OUT / "b3" / "docs" / doc["doc_id"] / "hypotheses.json"
hyp = json.loads(hyp_path.read_text())
nv = hyp["nonverbal"]
vb = hyp["verbal"]
print("nonverbal fields", len(nv), "marked", sum(1 for m in nv.values() if m["is_marked"]))
print("verbal sources", sorted({f["source"] for f in vb.values()}))
print("tube_edta", vb.get("tube_edta"))
assert vb.get("tube_edta", {}).get("source") != "prior_expected"


## 2. Annotated canvas


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

ann = OUT / "b3" / "docs" / doc["doc_id"] / "annotated_canvas.png"
show_rgb(ann, "Block 3 overlay", figsize=(12, 10))
download(OUT / "block3.zip")
